In [2]:
import os
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader

from cbir.dataset import CbirDataset
from cbir.models import DatabaseFeatureExtractor

In [7]:
BATCH_SIZE = 60
OXFORD_ROOT = "/home/ubuntu/data/datasets/roxford5k/jpg"
PARIS_ROOT = "/home/ubuntu/data/datasets/rparis6k/jpg"
CACHE_DIR = "/home/ubuntu/data/feature_cache"

In [8]:
oxford_dataset = CbirDataset(OXFORD_ROOT)
oxford_dataloader = DataLoader(oxford_dataset, batch_size=BATCH_SIZE)
print(f"Number of batches in roxford5k: {len(oxford_dataloader)}")

paris_dataset = CbirDataset(PARIS_ROOT)
paris_dataloader = DataLoader(paris_dataset, batch_size=BATCH_SIZE)
print(f"Number of batches in rparis6k: {len(paris_dataloader)}")

Number of batches in roxford5k: 85
Number of batches in rparis6k: 107


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
db_extractor = DatabaseFeatureExtractor(num_features=500, num_clusters=10).to(device)
db_extractor.eval()

DatabaseFeatureExtractor(
  (resnet_backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
 

In [10]:
with torch.no_grad():
    print(f"Processing roxford5k....")

    oxford_feature_cache = {}

    for batch_images, batch_ids in tqdm(oxford_dataloader):
        batch_images = batch_images.to(device)
        batched_features = db_extractor(batch_images)
        batched_features = batched_features.cpu()

        for i in range(len(batch_ids)):
            image_id = batch_ids[i]
            single_image_features = batched_features[i]            
            oxford_feature_cache[image_id] = single_image_features

    torch.save(oxford_feature_cache, os.path.join(CACHE_DIR, "roxford5k_features.pkl"))
    print(f"Features extracted successfully and saved to {CACHE_DIR}/roxford5k_features.pkl")

    print(f"Processing rparis6k....")

    paris_feature_cache = {}

    for batch_images, batch_ids in tqdm(paris_dataloader):
        batch_images = batch_images.to(device)
        batched_features = db_extractor(batch_images)
        batched_features = batched_features.cpu()

        for i in range(len(batch_ids)):
            image_id = batch_ids[i]
            single_image_features = batched_features[i]            
            paris_feature_cache[image_id] = single_image_features

    torch.save(oxford_feature_cache, os.path.join(CACHE_DIR, "rparis6k_features.pkl"))
    print(f"Features extracted successfully and saved to {CACHE_DIR}/rparis6k_features.pkl")

Processing roxford5k....


100%|██████████| 85/85 [09:56<00:00,  7.01s/it]


Features extracted successfully and saved to /home/ubuntu/data/feature_cache/roxford5k_features.pkl
Processing rparis6k....


 39%|███▉      | 42/107 [04:56<07:39,  7.06s/it]


RuntimeError: Unsupported image file. Only jpeg, png, webp and gif are currently supported. For avif and heic format, please rely on `decode_avif` and `decode_heic` directly.